# CS604 Colab launcher — Ana: kNN re-ranking on top of Van's best zoom result (no training)

Van's best score without balancing is her **full FT at 448 with crop 0.9 + zoom: 0.832** (`full_ft_20261017_122237`). This tests whether Ana's kNN re-ranking (+1.40 on her LoRA at 336) improves it: `scripts/predict_rerank.py --zoom` reproduces Van's zoom prediction exactly (checked file-for-file against `src.predict --zoom` at crop 0.9 and 0.75) and then re-ranks those zoom probabilities with each candidate's nearest training photos.

Van's models need Van's code, so this notebook **clones `van-dev`** and copies in only `scripts/predict_rerank.py` from `ana-branch`. Nothing on either branch changes.

One pass writes, in `CS604_RUNS/rerank_submission_van/<run>/val/`:
- `val_team_08.zip`: **zoom + kNN (beta 20)**, the main submission;
- `val_team_08_zoom.zip`: zoom only, should reproduce **0.832** (paired check);
- `val_team_08_plain.zip`: plain view only (0.825);
- `val_team_08_zoom_knn_b10/b40.zip`: other kNN weights, only if you want to spend submissions on them;
- `rerank_stats.json`: how many predictions each variant changed.

K = 5, m = 3, beta = 20 come from Ana's all-class holdout (LoRA at 336); there is no holdout for Van's checkpoints (trained on all 45 photos per class), so they are not re-tuned here. For her Task 2 model instead, set `RUN_NAME = "van_adaptformer_20261021_083823"` (zoom alone: 0.822).

About 50 min on an A100: data ~15 min, bank of 450,000 training photos at 448 px ~25–30 min (once, cached on Drive), predictions with zoom ~5 min. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "van-dev"  # Van's code builds her model
RERANK_FROM = "ana-branch"  # only scripts/predict_rerank.py is taken from here
RUN_NAME = "full_ft_20261017_122237"  # Van: full FT @448 (0.825 plain, 0.832 crop 0.9 + zoom)
# Task 2 instead: "van_adaptformer_20261021_083823" (0.812 plain, 0.822 crop 0.9 + zoom)
ZOOM, CROP_PCT = True, 0.9  # Van's best zoom setting at 448
BETAS = [20.0, 10.0, 40.0]  # first = main submission; fixed from Ana's all-class holdout
TEAM = "08"
SPLITS = ["val"]  # add "test" when the test split is released
K, M = 5, 3  # fixed on the all-class holdout; do not tune on CodaBench
OUT_NAME = "rerank_submission_van"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

In [ ]:
# take only the re-ranking script from ana-branch; Van's branch is not modified
!git -C {REPO_DIR} fetch origin {RERANK_FROM}
!git -C {REPO_DIR} show FETCH_HEAD:scripts/predict_rerank.py > {REPO_DIR}/scripts/predict_rerank.py
!head -3 {REPO_DIR}/scripts/predict_rerank.py


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Predict with re-ranking

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
assert (RUN_DIR / "last.pt").is_file(), f"no checkpoint in {RUN_DIR}"
OUT = Path(DRIVE_OUT) / OUT_NAME / RUN_NAME
BANK = OUT / f"bank_{RUN_NAME}.pt"
for split in SPLITS:
    if split == "test" and not ((Path(DATA_DIR) / "test.txt").is_file() or (Path(DATA_DIR) / "test").is_dir()):
        print("test split not available yet; skipping"); continue
    proc = subprocess.run([sys.executable, "-u", "scripts/predict_rerank.py", "--run_dir", str(RUN_DIR),
                           "--data_root", DATA_DIR, "--split", split, "--team", TEAM, "--out_dir", str(OUT / split),
                           "--bank", str(BANK), "--K", str(K), "--m", str(M), "--betas", *map(str, BETAS),
                           *(["--zoom", "--crop_pct", str(CROP_PCT)] if ZOOM else [])])
    if proc.returncode != 0:
        end_runtime(f"prediction failed (exit {proc.returncode})")
        raise SystemExit("prediction failed")
    extra = (["zoom"] + [f"zoom_knn_b{b:g}" for b in BETAS[1:]]) if ZOOM else [f"knn_b{b:g}" for b in BETAS[1:]]
    for name in [f"{split}_team_{TEAM}", f"{split}_team_{TEAM}_plain"] + [f"{split}_team_{TEAM}_{v}" for v in extra]:
        folder = OUT / split / name
        folder.mkdir(exist_ok=True)
        shutil.copy(OUT / split / f"{name}.txt", folder / f"{split}_team_{TEAM}.txt")  # CodaBench expects this file name
        shutil.make_archive(str(OUT / split / name), "zip", root_dir=folder, base_dir=f"{split}_team_{TEAM}.txt")
        print("zip:", OUT / split / f"{name}.zip")
    print(json.dumps(json.loads((OUT / split / "rerank_stats.json").read_text()), indent=2))


## 6. Submit

Submit `val_team_08.zip` (zoom + kNN) and `val_team_08_zoom.zip` (zoom only) from `rerank_submission_van/<run>/val/`. The zoom-only file should give Van's **0.832**; the difference is what kNN adds on top of her best. The plain file (0.825) and the beta 10 / 40 files are optional. Inside each zip the file is named `val_team_08.txt`.

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")